# SQL Business Analysis

## Project Purpose

This notebook uses SQL to analyse the missed refund dataset and answer practical business questions.

The analysis focuses on:

- missed refund volumes
- outstanding financial value
- department performance
- root causes
- case outcomes
- monthly trends

The purpose is to demonstrate how SQL can be used to turn operational data into clear business insights.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
database_path = Path("../sql/missed_refunds.db")

connection = sqlite3.connect(database_path)

print("Connected to the SQLite database.")

Connected to the SQLite database.


In [3]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, connection)

,name
0,missed_refunds


In [4]:
query = """
PRAGMA table_info(missed_refunds);
"""

pd.read_sql_query(query, connection)

,cid,name,type,notnull,dflt_value,pk
0,0,Case ID,TEXT,0,None,0
1,1,Policy Number,TEXT,0,None,0
2,2,Client Number,TEXT,0,None,0
3,3,Snapshot Date,TEXT,0,None,0
4,4,Outstanding Amount,REAL,0,None,0
5,5,Cancellation Status,TEXT,0,None,0
6,6,Cancelling Department,TEXT,0,None,0
7,7,Cancelling Agent,TEXT,0,None,0
8,8,Agent Working,TEXT,0,None,0
9,9,Refund Type,TEXT,0,None,0


## 1. Overall Case Volume and Financial Value

### Business Question

How many missed-refund cases are in the dataset, and what is their total and average outstanding value?

In [5]:
query = """
SELECT
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding,
    ROUND(AVG("Outstanding Amount"), 2) AS average_outstanding
FROM missed_refunds;
"""

overall_summary = pd.read_sql_query(query, connection)
overall_summary

,total_cases,total_outstanding,average_outstanding
0,2871,160215.8,55.8


### Business Insight

The dataset contains **2,871 missed-refund cases**, with a combined outstanding value of **£160,215.80**.

The average outstanding amount is **£55.80 per case**. This shows that missed refunds represent both a substantial operational workload and a significant financial liability. Improving the refund process could therefore reduce the case backlog while ensuring customers receive money owed to them more promptly.

## 2. Missed Refunds by Cancelling Department

### Business Question

Which cancelling departments account for the highest number and value of missed-refund cases?

In [6]:
query = """
SELECT
    "Cancelling Department" AS department,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding
FROM missed_refunds
GROUP BY "Cancelling Department"
ORDER BY total_cases DESC;
"""

department_summary = pd.read_sql_query(query, connection)
department_summary

,department,total_cases,total_outstanding
0,Retentions,1205,67754.46
1,Customer Service,1023,55224.05
2,Claims,453,25056.57
3,Complaints,190,12180.72


### Business Insight

**Retentions** accounts for the highest missed-refund volume, with **1,205 cases** and **£67,754.46 outstanding**. This is followed by **Customer Service**, with **1,023 cases** worth **£55,224.05**.

Together, these two departments account for approximately **78% of all missed-refund cases**, making them the strongest initial focus for further investigation and process improvement.

However, these figures show case volume rather than error rates. Retentions may have more missed refunds because it handles more cancellations overall. The total number of cancellations completed by each department would be required to compare departmental performance fairly.

## 3. Missed Refunds by Root Cause

### Business Question

What are the most common root causes of missed refunds, and how much outstanding value is associated with each cause?

In [7]:
query = """
SELECT
    "Root Cause" AS root_cause,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding
FROM missed_refunds
GROUP BY "Root Cause"
ORDER BY total_cases DESC;
"""

root_cause_summary = pd.read_sql_query(query, connection)
root_cause_summary

,root_cause,total_cases,total_outstanding
0,Payment Date Misunderstood,1329,74583.91
1,Agent Forgot,820,43811.35
2,Waiting for Information,409,22771.07
3,Refund Mailbox Delay,313,19049.47


### Business Insight

**Payment Date Misunderstood** is the leading root cause, accounting for **1,329 cases** and **£74,583.91** in outstanding refunds. This represents approximately **46% of all cases**.

The second most common cause is **Agent Forgot**, with **820 cases** and **£43,811.35** outstanding.

Together, these causes account for approximately **75% of missed-refund cases**. This suggests that targeted guidance on payment dates, supported by process reminders or system prompts, could address a substantial proportion of the missed-refund workload.

Cases caused by **Waiting for Information** and **Refund Mailbox Delay** may require different improvements, such as clearer ownership, follow-up timescales and monitoring of outstanding work.

## 4. Case Outcomes

### Business Question

How many reviewed cases resulted in a processed refund, and how many were found to require no action?

In [8]:
query = """
SELECT
    "Outcome" AS outcome,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding,
    ROUND(AVG("Outstanding Amount"), 2) AS average_outstanding
FROM missed_refunds
GROUP BY "Outcome"
ORDER BY total_cases DESC;
"""

outcome_summary = pd.read_sql_query(query, connection)
outcome_summary

,outcome,total_cases,total_outstanding,average_outstanding
0,Actioned,2359,134821.38,57.15
1,No Action,463,23395.96,50.53
2,Raised,49,1998.46,40.78


### Business Insight

Most reviewed cases resulted in a refund being processed. **2,359 cases (approximately 82%)** were marked as **Actioned**, representing **£134,821.38** in associated refund value.

A further **49 cases**, worth **£1,998.46**, were marked as **Raised**. These cases required either senior review or action by another department, creating a potential dependency that may delay completion and should be monitored.

**463 cases** were recorded as **No Action**. However, this category combines cases where the refund had already been processed with cases where no refund was due. Separating these into distinct outcomes would improve reporting and make it easier to identify false positives within the original snapshot.

## 5. Monthly Missed-Refund Trend

### Business Question

How do missed-refund case volumes and financial values change between monthly snapshots?

In [9]:
query = """
SELECT
    "Snapshot Date" AS snapshot_date,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding
FROM missed_refunds
GROUP BY "Snapshot Date"
ORDER BY "Snapshot Date";
"""

monthly_summary = pd.read_sql_query(query, connection)
monthly_summary

,snapshot_date,total_cases,total_outstanding
0,2025-01-31,195,11649.74
1,2025-02-28,139,7559.31
2,2025-03-31,133,7153.59
3,2025-04-30,177,10888.28
4,2025-05-31,160,8932.56
5,2025-06-30,170,9698.16
6,2025-07-31,179,9921.97
7,2025-08-31,142,7340.79
8,2025-09-30,161,8846.14
9,2025-10-31,145,7610.74


### Business Insight

Monthly missed-refund volumes vary throughout the period, ranging from **133 to 195 cases**.

The highest volume occurred in **January 2025**, with **195 cases** and **£11,649.74** in associated value. The lowest monthly volume was **133 cases**, recorded in both **March 2025** and **March 2026**.

These changes should be interpreted alongside the simulated business events built into the synthetic dataset, including periods of new-starter onboarding and targeted staff training. Higher volumes may reflect temporary knowledge gaps during onboarding, while later reductions may represent the intended effect of training interventions.

January recorded relatively high volumes in both years, which could suggest a recurring seasonal pattern. However, the dataset contains only two January observations, so more historical data would be required to assess seasonality reliably.

Because this is a synthetic dataset, these patterns demonstrate how an analyst could connect operational events with performance data; they do not represent findings about the real organisation.

## 6. Training Intervention Analysis

### Business Question

Did the simulated training intervention introduced in July 2025 reduce the proportion of cases caused by payment-date misunderstandings?

Cases from March 2026 onwards are excluded from this comparison because a separate new-starter scenario begins at that point.

In [10]:
query = """
SELECT
    CASE
        WHEN "Snapshot Date" < '2025-07-01' THEN 'Before Training'
        ELSE 'After Training'
    END AS training_period,
    COUNT(*) AS total_cases
FROM missed_refunds
WHERE "Snapshot Date" < '2026-03-01'
GROUP BY training_period
ORDER BY training_period DESC;
"""

training_period_totals = pd.read_sql_query(query, connection)
training_period_totals

,training_period,total_cases
0,Before Training,974
1,After Training,1306


In [11]:
query = """
SELECT
    CASE
        WHEN "Snapshot Date" < '2025-07-01' THEN 'Before Training'
        ELSE 'After Training'
    END AS training_period,

    COUNT(*) AS total_cases,

    SUM(
        CASE
            WHEN "Root Cause" = 'Payment Date Misunderstood' THEN 1
            ELSE 0
        END
    ) AS payment_date_cases,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Root Cause" = 'Payment Date Misunderstood' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS payment_date_percentage

FROM missed_refunds
WHERE "Snapshot Date" < '2026-03-01'
GROUP BY training_period
ORDER BY training_period DESC;
"""

training_comparison = pd.read_sql_query(query, connection)
training_comparison

,training_period,total_cases,payment_date_cases,payment_date_percentage
0,Before Training,974,543,55.7
1,After Training,1306,525,40.2


### Business Insight

Before the simulated training intervention, **55.7%** of missed-refund cases were caused by payment-date misunderstandings. After training, this decreased to **40.2%**.

This represents a reduction of **15.5 percentage points**, suggesting that the simulated training intervention had its intended effect on this root cause.

Although the number of payment-date cases only decreased from **543 to 525**, the post-training period contains more monthly snapshots and more cases overall. Comparing percentages therefore provides a fairer assessment than comparing raw totals.

Because the dataset is synthetic, this result validates that the intended business scenario is represented in the generated data. In a real operational setting, the same analysis could be used to evaluate whether training was associated with improved performance.

## 7. New-Starter Impact Analysis

### Business Question

Did the proportion of payment-date misunderstandings increase after the simulated new starters joined in March 2026?

In [12]:
query = """
SELECT
    CASE
        WHEN "Snapshot Date" < '2026-03-01' THEN 'Before New Starters'
        ELSE 'After New Starters'
    END AS starter_period,

    COUNT(*) AS total_cases,

    SUM(
        CASE
            WHEN "Root Cause" = 'Payment Date Misunderstood' THEN 1
            ELSE 0
        END
    ) AS payment_date_cases,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Root Cause" = 'Payment Date Misunderstood' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS payment_date_percentage

FROM missed_refunds
WHERE "Snapshot Date" >= '2025-07-01'
GROUP BY starter_period
ORDER BY starter_period DESC;
"""

new_starter_comparison = pd.read_sql_query(query, connection)
new_starter_comparison

,starter_period,total_cases,payment_date_cases,payment_date_percentage
0,Before New Starters,1306,525,40.2
1,After New Starters,591,261,44.2


### Business Insight

During the post-training period before the simulated new starters joined, payment-date misunderstandings accounted for **40.2%** of missed-refund cases.

After the new starters joined in March 2026, this increased to **44.2%**, representing a rise of **4.0 percentage points**. This suggests that the simulated onboarding period partially reversed the improvement associated with the earlier training intervention.

The result highlights a potential operational need for structured onboarding, refresher guidance and additional quality monitoring when new employees begin handling cancellations.

However, the post-new-starter period contains only four monthly snapshots. In a real analysis, additional months would be required to determine whether the increase represented a temporary onboarding effect or a sustained change in performance.

Because the dataset is synthetic, this analysis demonstrates how operational data could be used to monitor performance following workforce changes rather than establishing a real causal relationship.

## 8. Seasonal Mailbox Delay Analysis

### Business Question

Are refund mailbox delays more common during the simulated high-workload months of December and January?

In [13]:
query = """
SELECT
    CASE
        WHEN strftime('%m', "Snapshot Date") IN ('12', '01')
            THEN 'December and January'
        ELSE 'Other Months'
    END AS seasonal_period,

    COUNT(*) AS total_cases,

    SUM(
        CASE
            WHEN "Root Cause" = 'Refund Mailbox Delay' THEN 1
            ELSE 0
        END
    ) AS mailbox_delay_cases,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Root Cause" = 'Refund Mailbox Delay' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS mailbox_delay_percentage

FROM missed_refunds
GROUP BY seasonal_period
ORDER BY mailbox_delay_percentage DESC;
"""

seasonal_comparison = pd.read_sql_query(query, connection)
seasonal_comparison

,seasonal_period,total_cases,mailbox_delay_cases,mailbox_delay_percentage
0,December and January,561,94,16.8
1,Other Months,2310,219,9.5


### Business Insight

Refund mailbox delays accounted for **16.8%** of missed-refund cases during December and January, compared with **9.5%** during other months.

This represents an increase of **7.3 percentage points**, supporting the simulated scenario that seasonal workload contributes to greater mailbox delays during these months.

In a real operational environment, this pattern could support earlier resource planning, temporary redistribution of work, clearer mailbox ownership and closer backlog monitoring ahead of expected high-workload periods.

The dataset contains only three December or January snapshots, so additional years would be needed to establish a reliable recurring seasonal trend. As this dataset is synthetic, the result confirms that the intended seasonal scenario is reflected in the generated data.

## 9. Customer Contact and Financial Exposure

### Business Question

How many missed-refund cases involve customers who have already contacted the business, and what financial value is associated with those cases?

In [14]:
query = """
SELECT
    "Customer Contacted" AS customer_contacted,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding,
    ROUND(AVG("Outstanding Amount"), 2) AS average_outstanding
FROM missed_refunds
GROUP BY "Customer Contacted"
ORDER BY total_cases DESC;
"""

customer_contact_summary = pd.read_sql_query(query, connection)
customer_contact_summary

,customer_contacted,total_cases,total_outstanding,average_outstanding
0,No,2158,122623.5,56.82
1,Yes,713,37592.3,52.72


### Business Insight

Customers had already contacted the business in **713 cases**, representing approximately **24.8% of the total missed-refund workload**. These cases were associated with **£37,592.30** in outstanding value.

The average amount was slightly lower for contacted customers (**£52.72**) than for customers who had not contacted the business (**£56.82**). This suggests that customer contact was not driven only by higher-value refunds.

Cases involving prior customer contact may require additional prioritisation because the customer is already aware of the issue. Delays could increase repeat contact, complaints and the risk of a poor customer outcome.

A useful process improvement would be to include customer-contact status within case-prioritisation rules rather than prioritising cases solely by financial value.

## 10. Missed Refunds by Refund Type

### Business Question

Which refund types account for the highest missed-refund volumes and financial values?

In [15]:
query = """
SELECT
    "Refund Type" AS refund_type,
    COUNT(*) AS total_cases,
    ROUND(SUM("Outstanding Amount"), 2) AS total_outstanding,
    ROUND(AVG("Outstanding Amount"), 2) AS average_outstanding
FROM missed_refunds
GROUP BY "Refund Type"
ORDER BY total_cases DESC;
"""

refund_type_summary = pd.read_sql_query(query, connection)
refund_type_summary

,refund_type,total_cases,total_outstanding,average_outstanding
0,Death of Pet,1559,90280.60,57.91
1,Cancellation Before Premium Due,735,38971.28,53.02
2,Cancellation from Renewal (Void),577,30963.92,53.66


### Business Insight

**Death of Pet** is the most common refund type in the dataset, accounting for **1,559 cases**, or approximately **54% of the total missed-refund workload**. These cases are associated with **£90,280.60**, representing approximately **56% of the total outstanding value**.

This category also has the highest average outstanding amount at **£57.91 per case**. Because these cases follow the death of a customer’s pet, delays may create additional customer-outcome and reputational risk beyond the financial value involved.

However, the dataset does not contain the total number of cancellations completed for each refund type. The results therefore show the distribution of missed-refund cases, not the missed-refund rate for each cancellation scenario.

## Key Findings

- The dataset contains **2,871 missed-refund cases** with a combined outstanding value of **£160,215.80** and an average value of **£55.80 per case**.

- **Retentions and Customer Service** together account for approximately **78% of all cases**, making them the main areas for further investigation. However, total cancellation volumes would be required to compare departmental error rates fairly.

- **Payment Date Misunderstood** is the leading root cause, accounting for approximately **46% of cases**. Together with **Agent Forgot**, it represents approximately **75% of the missed-refund workload**.

- Approximately **82% of reviewed cases** resulted in a refund being processed. The existing **No Action** outcome combines two different situations and should be separated to improve reporting accuracy.

- Following the simulated training intervention, payment-date misunderstandings decreased from **55.7% to 40.2%**, indicating that the intended improvement is represented in the synthetic data.

- After simulated new starters joined, payment-date misunderstandings increased to **44.2%**, highlighting how onboarding and workforce changes could affect operational performance.

- Refund mailbox delays increased from **9.5% in other months to 16.8% during December and January**, supporting the simulated seasonal-workload scenario.

- Customers had already contacted the business in approximately **24.8% of cases**, indicating that customer-contact status could be used alongside financial value when prioritising work.

- **Death of Pet** cases represent approximately **54% of missed-refund volume** and **56% of associated value**, creating an additional customer-outcome risk due to the sensitive circumstances involved.

These findings are based on a synthetic dataset designed to represent realistic operational scenarios. They demonstrate how SQL could be used to identify process issues, assess simulated interventions and support data-informed business decisions.

## Recommendations and Limitations

### Recommendations

1. **Provide targeted payment-date training**  
   Focus guidance on identifying when refunds are due, particularly for cancellation scenarios where payment dates may be misunderstood.

2. **Strengthen new-starter onboarding**  
   Include refund-process training, practical examples and short-term quality monitoring to reduce knowledge gaps during onboarding.

3. **Plan for seasonal mailbox demand**  
   Review staffing, mailbox ownership and backlog monitoring ahead of December and January, when delays may be more likely.

4. **Introduce risk-based case prioritisation**  
   Prioritisation should consider whether the customer has already made contact, the refund value, the age of the case and the sensitivity of the cancellation circumstances.

5. **Monitor raised cases separately**  
   Cases requiring senior approval or another department should have clear ownership and expected completion times to prevent avoidable delays.

6. **Improve outcome classifications**  
   Split `No Action` into separate categories such as `Refund Already Processed` and `No Refund Due` to improve reporting and identify false positives in the original snapshot.

### Limitations

- The dataset is synthetic and demonstrates analytical techniques rather than measuring the performance of a real organisation.

- Departmental results show missed-refund volumes, not error rates. Total cancellation volumes by department would be required for a fair performance comparison.

- The data contains monthly snapshots but does not currently track changes to individual cases between snapshots.

- Case creation, assignment, escalation and completion dates are not available, so processing times and service-level performance cannot yet be measured.

- The training, new-starter and seasonal effects were intentionally built into the data-generation process. The analysis confirms that these simulated patterns are present but does not establish real-world causation.

- More historical data would be required to assess seasonality and longer-term trends reliably.

### Next Development Phase

The next phase will extend the project from a one-off monthly analysis into a simulated operational reporting workflow. A weekly status feed will track new, ageing, dependency and completed cases, allowing future analysis of backlog movement, resolution times and operational performance.